<a href="https://colab.research.google.com/github/Yami2912/HER360/blob/feature%2Fhealthcare/HER360-Healthcare-ML/notebooks/fertility/ovulation_predictor/02_prepare_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import userdata
import os
TOKEN = userdata.get("GITHUB_TOKEN")
GH_USER, REPO = "Yami2912", "HER360"

os.chdir("/content")
if not os.path.exists(f"/content/{REPO}"):
    !git clone https://{TOKEN}@github.com/{GH_USER}/{REPO}.git

%cd /content/HER360
!git config user.name  "Yami2912"
!git config user.email "yamipatel0618@gmail.com"
!git fetch origin
!git checkout feature/healthcare
!git pull origin feature/healthcare
!git log --oneline -3

In [ ]:
import pathlib, shutil, zipfile, sys, json
BASE = pathlib.Path("/content/HER360/HER360-Healthcare-ML")
assert BASE.exists(), "HER360-Healthcare-ML not found - are you on feature/healthcare?"

folders = [
    "datasets/fertility/ovulation_predictor/raw",
    "datasets/fertility/ovulation_predictor/processed",
    "datasets/fertility/dual_sync",
    "datasets/fertility/loss_sensitive_protocol",
    "datasets/fertility/personalised_ai_care",
    "datasets/fertility/postpartum_support",
    "integration/fertility",
    "models/fertility/ovulation_predictor",
    "notebooks/fertility/ovulation_predictor",
    "preprocessing/fertility",
    "tests/fertility",
]
for f in folders:
    p = BASE / f
    p.mkdir(parents=True, exist_ok=True)
    if not any(p.iterdir()):
        (p / ".gitkeep").touch()
(BASE / "preprocessing/fertility/__init__.py").touch()
print("folders ready")

In [ ]:
%%writefile /content/HER360/HER360-Healthcare-ML/preprocessing/fertility/ovulation_prep.py
"""
HER360 Fertility - Ovulation Predictor: data preparation (v1)

Turns the raw source-labelled candidate file into a model-ready table
WITHOUT changing any label and WITHOUT inventing any value.

Corrections made (each is logged):
  1. Sort rows by participant and cycle_number.
  2. Remove repeats of (participant, cycle_number): keep first occurrence in file order.
  3. Drop columns not known at prediction time (leakage audit).
  4. Build features from the participant's PREVIOUS cycles only.
  5. Label how much history each row has.
Not done on purpose: no label edits, no imputation, no changing the splits.
"""
import json
import numpy as np
import pandas as pd

TARGET = "target_estimated_ovulation_day"
GROUP = "participant_group_id"
SPLIT = "split"

LEAKAGE_COLUMNS = {
    "cycle_length_days": "Length of the CURRENT cycle - unknown until the cycle ends (kept only to build lags of later cycles)",
    "first_day_high_fertility_sign": "Observed during the cycle; not guaranteed at prediction time",
    "total_high_days": "Accumulated over time; post-start information",
    "high_days_post_peak": "Post-peak information",
    "total_peak_days": "Accumulated / after-event information",
}

N_LAGS = 3


def load_raw(path):
    return pd.read_csv(path)


def prepare(df_raw):
    """Return (model_table, log_dict)."""
    log = {"rows_in": int(len(df_raw))}

    # repeated (participant, cycle_number): keep first occurrence in ORIGINAL file order
    df = df_raw.reset_index().rename(columns={"index": "_orig_row"})
    df = df.sort_values([GROUP, "cycle_number", "_orig_row"], kind="stable")
    repeated = df[df.duplicated([GROUP, "cycle_number"], keep="first")]
    log["repeated_rows_removed"] = int(len(repeated))
    log["repeated_rows_removed_original_row_numbers"] = repeated["_orig_row"].tolist()
    df = df.drop_duplicates([GROUP, "cycle_number"], keep="first").reset_index(drop=True)

    # lag features from PREVIOUS cycles of the same participant only
    length_lookup = df.set_index([GROUP, "cycle_number"])["cycle_length_days"]
    for k in range(1, N_LAGS + 1):
        keys = pd.MultiIndex.from_arrays([df[GROUP], df["cycle_number"] - k])
        df[f"prev_cycle_length_{k}"] = length_lookup.reindex(keys).to_numpy()

    lag_cols = [f"prev_cycle_length_{k}" for k in range(1, N_LAGS + 1)]
    df["n_prev_cycles"] = df[lag_cols].notna().sum(axis=1)
    df["prev_cycle_mean"] = df[lag_cols].mean(axis=1)
    df["prev_cycle_std"] = df[lag_cols].std(axis=1)  # NaN when fewer than 2 previous cycles

    # history status - a PROJECT convention (not a clinical rule)
    df["history_status"] = np.select(
        [df["n_prev_cycles"] == 0, df["n_prev_cycles"] < N_LAGS],
        ["insufficient", "limited"],
        default="sufficient",
    )

    keep = [GROUP, SPLIT, "cycle_number", "reproductive_category_code",
            *lag_cols, "n_prev_cycles", "prev_cycle_mean", "prev_cycle_std",
            "history_status", TARGET]
    out = df[keep].copy()

    log["rows_out"] = int(len(out))
    log["history_status_counts"] = out["history_status"].value_counts().to_dict()
    log["history_status_by_split"] = (
        out.groupby([SPLIT, "history_status"]).size().unstack(fill_value=0).to_dict("index")
    )
    log["labels_modified"] = 0
    log["values_imputed"] = 0
    return out, log


FEATURES_V1 = [
    "prev_cycle_length_1", "prev_cycle_length_2", "prev_cycle_length_3",
    "prev_cycle_mean", "prev_cycle_std", "n_prev_cycles",
]


def validate(out):
    """Hard checks - raise if anything is wrong."""
    assert out[TARGET].notna().all(), "missing target"
    assert not out.duplicated([GROUP, "cycle_number"]).any(), "repeated participant+cycle remain"
    assert (out.groupby(GROUP)[SPLIT].nunique() == 1).all(), "participant in >1 split"
    for c in FEATURES_V1:
        assert c not in LEAKAGE_COLUMNS and c not in (GROUP, SPLIT, TARGET)
    lagc = [c for c in out.columns if c.startswith("prev_cycle_length_")]
    assert out[lagc].stack().dropna().between(15, 60).all(), "implausible previous cycle length"
    return True

In [ ]:
%%writefile /content/HER360/HER360-Healthcare-ML/tests/fertility/test_ovulation_prep.py
import sys, pathlib
import pandas as pd
import pytest

# find HER360-Healthcare-ML root by walking up until preprocessing/fertility exists
ROOT = next(p for p in pathlib.Path(__file__).resolve().parents if (p / "preprocessing" / "fertility").exists())
sys.path.insert(0, str(ROOT / "preprocessing" / "fertility"))
import ovulation_prep as op


def make(rows):
    cols = ["cycle_number", "reproductive_category_code", "cycle_length_days",
            "first_day_high_fertility_sign", "total_high_days", "high_days_post_peak",
            "total_peak_days", op.TARGET, op.GROUP, op.SPLIT]
    return pd.DataFrame(rows, columns=cols)


def row(cn, length, target, g="p1", split="train"):
    return (cn, 0, length, 10.0, 2.0, 0.0, 2.0, target, g, split)


def test_lags_use_only_previous_cycles():
    t, _ = op.prepare(make([row(1, 28, 14), row(2, 30, 15), row(3, 26, 13)]))
    r3 = t[t.cycle_number == 3].iloc[0]
    assert r3.prev_cycle_length_1 == 30 and r3.prev_cycle_length_2 == 28
    r1 = t[t.cycle_number == 1].iloc[0]
    assert r1.n_prev_cycles == 0 and r1.history_status == "insufficient"


def test_gap_in_cycle_numbers_is_not_bridged():
    t, _ = op.prepare(make([row(1, 28, 14), row(3, 30, 15)]))  # cycle 2 missing
    r = t[t.cycle_number == 3].iloc[0]
    assert pd.isna(r.prev_cycle_length_1)
    assert r.prev_cycle_length_2 == 28


def test_repeated_cycle_removed_first_kept_labels_untouched():
    t, log = op.prepare(make([row(1, 28, 14), row(2, 27, 13), row(2, 26, 13)]))
    assert log["repeated_rows_removed"] == 1
    assert len(t) == 2
    assert t[t.cycle_number == 2].iloc[0][op.TARGET] == 13


def test_lags_do_not_cross_participants():
    t, _ = op.prepare(make([row(1, 28, 14, "a"), row(2, 30, 15, "b")]))
    assert pd.isna(t[t[op.GROUP] == "b"].iloc[0].prev_cycle_length_1)


def test_leakage_columns_absent_from_output_and_features():
    t, _ = op.prepare(make([row(1, 28, 14)]))
    for c in op.LEAKAGE_COLUMNS:
        assert c not in t.columns
        assert c not in op.FEATURES_V1


def test_validate_catches_split_leakage():
    t, _ = op.prepare(make([row(1, 28, 14, "a", "train"), row(2, 28, 14, "a", "test")]))
    with pytest.raises(AssertionError):
        op.validate(t)

In [ ]:
from google.colab import files

RAW = BASE / "datasets/fertility/ovulation_predictor/raw"
RAW_FILE = RAW / "ovulation_source_candidate_with_audit_columns.csv"

if not RAW_FILE.exists():
    print("Raw file not found in repo - upload HER360_Separate_Feature_Datasets.zip")
    up = files.upload()
    zname = list(up.keys())[0]
    zipfile.ZipFile(zname).extractall("/content/scratch_datasets")
    SRC = pathlib.Path("/content/scratch_datasets/01_Ovulation_Predictor")
    for f in ["ovulation_source_candidate_with_audit_columns.csv",
              "ovulation_minimal_prediction_time_candidate.csv",
              "ovulation_feature_leakage_audit.csv"]:
        shutil.copy(SRC / f, RAW / f)
print("raw file present:", RAW_FILE.exists())

In [ ]:
sys.path.insert(0, str(BASE / "preprocessing/fertility"))
import importlib, ovulation_prep as op
importlib.reload(op)

table, log = op.prepare(op.load_raw(RAW_FILE))
op.validate(table)

OUT = BASE / "datasets/fertility/ovulation_predictor/processed"
table.to_csv(OUT / "ovulation_model_table_v1.csv", index=False)
(OUT / "ovulation_prep_log_v1.json").write_text(json.dumps(log, indent=2, default=str))

print(json.dumps(log, indent=2, default=str))
display(table.head(8))

In [ ]:
!pip install -q pytest
%cd /content/HER360/HER360-Healthcare-ML
!python -m pytest tests/fertility -v

In [ ]:
note = '''# Ovulation predictor dataset notes (v1)

Status: college major project use only. Not validated for clinical or production use.

- raw/: source file as supplied (do not edit).
- processed/ovulation_model_table_v1.csv: output of preprocessing/fertility/ovulation_prep.py
- processed/ovulation_prep_log_v1.json: what the preparation changed.

Known limits
- Label (estimated ovulation day) is source-estimated, not independently confirmed.
- ~157 participants; about 96% of rows share one reproductive category code (meaning undocumented).
- Cannot validate performance for irregular cycles or PCOS.
- Licence / provenance of the source data still to be confirmed.
- Before any real deployment: replace with a professionally verified dataset.
'''
(BASE / "datasets/fertility/ovulation_predictor/README.md").write_text(note)
print("written")

In [ ]:
req = BASE / "requirements.txt"
print("exists:", req.exists(), "| size in bytes:", req.stat().st_size if req.exists() else None)

existing = req.read_text() if req.exists() else ""
needed = ["pandas", "numpy", "scikit-learn", "matplotlib", "joblib", "pytest"]
have = {l.strip().split("==")[0].split(">=")[0].lower()
        for l in existing.splitlines() if l.strip() and not l.startswith("#")}
missing = [p for p in needed if p.lower() not in have]
print("already listed:", sorted(have))
print("adding:", missing)

if missing:
    with open(req, "a") as f:
        if existing and not existing.endswith("\n"):
            f.write("\n")
        f.write("\n".join(missing) + "\n")

print("\n--- requirements.txt now ---")
print(req.read_text())

In [ ]:
%cd /content/HER360
!git add HER360-Healthcare-ML
!git status --short

In [ ]:
import pathlib
BASE = pathlib.Path("/content/HER360/HER360-Healthcare-ML")

%cd /content/HER360
!git log --oneline -2          # should show 8ed57b4 on top

gi = BASE / ".gitignore"
lines = ["__pycache__/", "*.pyc", ".ipynb_checkpoints/"]
existing = gi.read_text() if gi.exists() else ""
add = [l for l in lines if l not in existing.splitlines()]
if add:
    with open(gi, "a") as f:
        if existing and not existing.endswith("\n"):
            f.write("\n")
        f.write("\n".join(add) + "\n")

!git rm -r --cached -q --ignore-unmatch HER360-Healthcare-ML/preprocessing/fertility/__pycache__ HER360-Healthcare-ML/tests/fertility/__pycache__
!git add HER360-Healthcare-ML/.gitignore
!git commit --amend --no-edit
!git show --stat --oneline HEAD

In [ ]:
from google.colab import userdata
TOKEN = userdata.get("GITHUB_TOKEN")
!git remote set-url origin https://{TOKEN}@github.com/Yami2912/HER360.git
!git push origin feature/healthcare

In [ ]:
%cd /content/HER360
!git ls-remote --heads origin

In [ ]:
%cd /content/HER360
!git fetch origin
print("--- default branch on GitHub ---")
!git ls-remote --symref origin HEAD
print("\n--- what main has that develop does not ---")
!git log --oneline origin/develop..origin/main
print("\n--- who made the newest commits on main ---")
!git log -5 --format="%h  %an  %ar  %s" origin/main